# Lecture 10 - Error Handling and Defensive Programming

## Learning Objectives

- Recognize and handle common Python exceptions
- Use try/except blocks to catch and handle errors gracefully
- Catch specific exceptions for targeted error handling
- Leverage else and finally clauses
- Raise exceptions and create custom exception classes
- Use assertions for debugging and preconditions

## Key Topics

- Common exception types: TypeError, ValueError, KeyError, FileNotFoundError
- try/except blocks
- Catching specific exceptions
- else and finally clauses
- raise and custom exception classes
- Assertions for debugging

## Common Exception Types

Python raises exceptions when it encounters an error during execution. Knowing the common built-in exceptions helps you diagnose and handle problems quickly:

- **TypeError**: Raised when an operation is applied to an object of inappropriate type.
- **ValueError**: Raised when a function receives an argument with the right type but an inappropriate value.
- **KeyError**: Raised when a dictionary key is not found.
- **FileNotFoundError**: Raised when trying to open a file that doesn't exist.

Recognizing these exceptions is the first step toward writing robust code that fails gracefully.

In [ ]:
# TypeError examples
try:
    result = "5" + 10
except TypeError as e:
    print(f"TypeError: {e}")

# ValueError example
try:
    num = int("not_a_number")
except ValueError as e:
    print(f"ValueError: {e}")

# KeyError example
data = {"name": "Alice", "age": 30}
try:
    print(data["occupation"])
except KeyError as e:
    print(f"KeyError: Key '{e}' not found in dictionary")

# FileNotFoundError example
try:
    with open("nonexistent_file.csv") as f:
        content = f.read()
except FileNotFoundError as e:
    print(f"FileNotFoundError: {e}")


## Try/Except Blocks

The `try/except` block is Python's primary mechanism for handling exceptions. Code that might raise an error goes inside the `try` block. If an exception occurs, execution jumps to the matching `except` block, preventing the program from crashing.

This is especially important in data science when processing messy real-world data — a single malformed row should not bring down the entire pipeline.

In [ ]:
# Basic try/except
def safe_parse_int(value):
    try:
        return int(value)
    except:
        return None

values = ["42", "3.14", "abc", "100", "12.5xyz"]
parsed = [safe_parse_int(v) for v in values]
print(f"Parsed integers: {parsed}")


## Catching Specific Exceptions

Always catch the most specific exception types rather than using a bare `except:`. This prevents you from accidentally swallowing unexpected errors like `KeyboardInterrupt` or `MemoryError`. You can also chain multiple `except` clauses to handle different exception types in different ways.

Specific exception handling makes your code's intent clear and helps with debugging.

In [ ]:
def divide_numbers(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero!")
        return None
    except TypeError as e:
        print(f"Type error: {e}")
        return None
    else:
        print(f"Division successful: {result}")
        return result
    finally:
        print("Division attempt finished.")

print("Result:", divide_numbers(10, 2))
print("---")
print("Result:", divide_numbers(10, 0))
print("---")
print("Result:", divide_numbers(10, "x"))


## `else` and `finally` Clauses

The `else` clause runs only if no exception was raised in the `try` block. It's useful for code that should execute only on success. The `finally` clause runs **unconditionally** — whether an exception occurred or not — making it ideal for cleanup actions like closing files or releasing resources.

Together, these clauses give you fine-grained control over the flow of error-prone code.

In [ ]:
def read_config_file(filepath):
    try:
        f = open(filepath, "r")
        content = f.read()
    except FileNotFoundError:
        print("Config file not found. Using defaults.")
        return {"theme": "light", "font_size": 12}
    except PermissionError:
        print("Permission denied. Using defaults.")
        return {"theme": "light", "font_size": 12}
    else:
        print("Config file loaded successfully.")
        # parse simple key=value lines
        config = {}
        for line in content.strip().split("\n"):
            if "=" in line:
                k, v = line.split("=", 1)
                config[k.strip()] = v.strip()
        return config
    finally:
        try:
            f.close()
        except NameError:
            pass

config = read_config_file("config.txt")
print(config)


## `raise` and Custom Exception Classes

Sometimes you need to signal that something is wrong in your own code. You can `raise` an exception at any point. Python also lets you define **custom exception classes** by inheriting from `Exception` (or a subclass of it). This is invaluable for building domain-specific validation logic.

For example, in a data pipeline you might raise a `DataValidationError` when a column contains unexpected values, making the failure reason crystal clear.

In [ ]:
# Defining a custom exception
class DataValidationError(Exception):
    """Raised when data fails validation checks."""
    pass

def validate_age(age):
    if not isinstance(age, (int, float)):
        raise DataValidationError(f"Age must be a number, got {type(age).__name__}")
    if age < 0 or age > 150:
        raise DataValidationError(f"Age {age} is out of valid range (0-150)")
    return True

# Test the validation
for value in [25, -5, "thirty", 101]:
    try:
        validate_age(value)
        print(f"Age {value}: valid")
    except DataValidationError as e:
        print(f"Validation failed for {value}: {e}")


## Assertions for Debugging

An `assert` statement checks that a condition is `True`. If it is `False`, Python raises an `AssertionError`. Assertions are a lightweight debugging tool — they document invariants and catch bugs early during development.

Use assertions for conditions that **should always be true** if your code is correct. Do **not** use assertions for input validation in production code (use regular `if` checks and raise exceptions instead), because assertions can be disabled with the `-O` flag.

In [ ]:
def normalize_scores(scores):
    """Scale scores to 0-1 range."""
    min_s, max_s = min(scores), max(scores)
    # Precondition: we have at least one score
    assert len(scores) > 0, "Scores list is empty"
    # Precondition: range is positive
    assert max_s > min_s, "All scores are identical (cannot normalize)"
    normalized = [(s - min_s) / (max_s - min_s) for s in scores]
    # Postcondition: all values in [0, 1]
    assert all(0 <= v <= 1 for v in normalized), "Normalization out of range"
    return normalized

print(normalize_scores([10, 20, 30, 40, 50]))


## Data Science Connection

Real-world data is messy. Files are missing, columns contain invalid types, and values fall outside expected ranges. Defensive programming — using `try/except`, custom exceptions, and assertions — ensures your data pipeline can handle these problems without crashing. This builds trust and reliability in your analysis and is a hallmark of production-grade data science code.